# LedgerLock: AP Invoice Fraud Detection Demo

This notebook demonstrates the LedgerLock AP fraud detection pipeline for logistics and cold-chain shippers. We'll:

1. Load and explore sample invoice data
2. Apply fraud detection rules
3. Analyze and visualize results
4. Export findings to Excel

In [ ]:
# Import required libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src import parser, rules, report

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

## 1. Load and Explore Sample Data

Let's load our sample data files and examine their contents.

In [ ]:
# Load sample data
invoices = parser.load_invoices('../data/invoices_sample.csv')
vendors = parser.load_vendors('../data/vendor_master.csv')
bols = parser.load_bols('../data/bol_sample.csv')

print("Sample Invoices:")
display(invoices.head())
print("\nVendor Master:")
display(vendors)
print("\nBOL Records:")
display(bols)

# Show basic statistics
print("\nBasic Invoice Statistics:")
print(f"Total Invoices: {len(invoices)}")
print(f"Total Vendors: {len(vendors)}")
print(f"Date Range: {invoices['invoice_date'].min()} to {invoices['invoice_date'].max()}")
print(f"Total Invoice Amount: ${invoices['amount'].sum():,.2f}")

## 2. Apply Fraud Detection Rules

Now we'll run our suite of fraud detection rules to identify suspicious invoices.

In [ ]:
# Run all fraud checks
flags = []

# 1. Check for duplicates
dup_flags = rules.flag_duplicates(invoices)
flags.append(dup_flags)
print(f"Found {len(dup_flags)} duplicate invoices")

# 2. Check for bank mismatches
bank_flags = rules.flag_bank_mismatch(invoices, vendors)
flags.append(bank_flags)
print(f"Found {len(bank_flags)} bank account mismatches")

# 3. Check for unknown vendors
vendor_flags = rules.flag_unknown_vendor(invoices, vendors)
flags.append(vendor_flags)
print(f"Found {len(vendor_flags)} unknown vendors")

# 4. Check for unusual amounts
amount_flags = rules.flag_unusual_amounts(invoices, vendors)
flags.append(amount_flags)
print(f"Found {len(amount_flags)} unusual amounts")

# 5. Check for BOL mismatches
bol_flags = rules.flag_bol_mismatch(invoices, bols)
flags.append(bol_flags)
print(f"Found {len(bol_flags)} BOL mismatches")

# Combine all flags
all_flags = pd.concat(flags, ignore_index=True)
print(f"\nTotal flags raised: {len(all_flags)}")
display(all_flags)

## 3. Analyze and Visualize Results

Let's create some visualizations to better understand the fraud detection results.

In [ ]:
# 1. Flag distribution
flag_counts = all_flags['reason'].value_counts()
fig1 = px.bar(x=flag_counts.index, 
              y=flag_counts.values,
              title='Distribution of Fraud Flags',
              labels={'x': 'Flag Type', 'y': 'Count'})
fig1.show()

# 2. Amount distribution by vendor
fig2 = px.box(invoices, 
              x='vendor_name', 
              y='amount',
              title='Invoice Amount Distribution by Vendor')
fig2.show()

# 3. Flags timeline
invoices_with_flags = invoices.merge(all_flags[['invoice_id', 'reason']], 
                                   on='invoice_id', 
                                   how='left')
flags_by_date = invoices_with_flags.groupby('invoice_date')['reason'].count()
fig3 = px.line(x=flags_by_date.index, 
               y=flags_by_date.values,
               title='Fraud Flags Over Time',
               labels={'x': 'Date', 'y': 'Number of Flags'})
fig3.show()

## 4. Export Results

Finally, let's save our findings to an Excel report for further analysis.

In [ ]:
# Export to Excel with multiple sheets
with pd.ExcelWriter('../data/fraud_analysis_report.xlsx') as writer:
    # Flags sheet
    all_flags.to_excel(writer, sheet_name='Fraud_Flags', index=False)
    
    # Summary statistics
    summary_stats = pd.DataFrame({
        'Metric': [
            'Total Invoices',
            'Total Vendors',
            'Total Flags',
            'Flag Rate',
            'Total Invoice Amount',
            'Flagged Invoice Amount',
        ],
        'Value': [
            len(invoices),
            len(vendors),
            len(all_flags),
            f"{len(all_flags)/len(invoices)*100:.1f}%",
            f"${invoices['amount'].sum():,.2f}",
            f"${invoices[invoices['invoice_id'].isin(all_flags['invoice_id'])]['amount'].sum():,.2f}"
        ]
    })
    summary_stats.to_excel(writer, sheet_name='Summary', index=False)
    
    # Flag distribution
    flag_counts.to_frame('Count').to_excel(writer, sheet_name='Flag_Distribution')

print("Report saved to data/fraud_analysis_report.xlsx")

# Display summary statistics
display(summary_stats)

## Next Steps

Based on this analysis, we can:

1. **Investigate High-Risk Invoices**:
   - Focus on invoices with multiple flags
   - Review unusual amount patterns by vendor

2. **Enhance Detection Rules**:
   - Add weekend/holiday invoice detection
   - Implement accessorial charge analysis
   - Add fuel surcharge validation

3. **Improve Reporting**:
   - Add risk scores for each flag
   - Create vendor risk profiles
   - Generate automated alerts for high-risk invoices

4. **Build Web Dashboard**:
   - Real-time fraud detection
   - Interactive visualizations
   - User-configurable thresholds